## AAL calculation

#### The code is modified to calculate AAL for Turkey and Brush Creek probabilistic modeling

In [1]:
import os
import glob
import pathlib as pl

import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal

In [2]:
def getTifData(tif_path: str):
    """Read a WSE raster and return the gdal objects"""
    src = gdal.Open(tif_path)
    rb = src.GetRasterBand(1)
    gt = src.GetGeoTransform()
    proj = src.GetProjection()
    return rb, gt, src

In [3]:
def query(x: float, y: float, gt: any, rb: any) -> float:
    """Queries one specific cell in the rasterband given an x, y in the 
    geotransform
    """
    px = int((x-gt[0]) / gt[1])   
    py = int((y-gt[3]) / gt[5])   
    return rb.ReadAsArray(px,py,1,1)[0][0]

In [4]:
def StructureData(gdf_path, gt, rb):
    """Read structures, compute WSE values, and depth values"""

    gdf = gpd.read_file(gdf_path) 
    # Add a new column for WSE values
    gdf['wse'] = None
    
    for i, idx in enumerate(gdf.index):
        x = gdf.loc[idx, 'x_sp']
        y = gdf.loc[idx, 'y_sp']
        pixel_value = query(x, y, gt, rb)
        
        # Assign WSE value to the GeoDataFrame, if value is -9999 it assigns the same -9999 value
        gdf.at[idx, 'wse'] = -9999 if pixel_value == -9999 else pixel_value

        # Calculating actual depth of water based on wse, ground elevation and foundation height. If the wse value is -9999, the depth is also assigned as -9999
        gdf.at[idx,'depth'] = gdf.at[idx,'wse'] - gdf.at[idx,'ground_elv'] - gdf.at[idx,'found_ht'] if gdf.at[idx,'wse'] != -9999 else -9999
    return gdf

In [5]:
def new_occtype(row, occtype_val, found_type_val):
    """Reads the  occupancy type and foundation type for RES3 occupancy types and reclassifies based on whether a structure has basement or not.
    Since RES3A to RES3F needed to be reclassified on their availability of the basement.
    """
    if row['occtype'] in occtype_val and row['found_type'] == found_type_val:
        return f"{row['occtype']}-{row['found_type']}"
    elif row['occtype'] in occtype_val and row['found_type'] != found_type_val:
        return f"{row['occtype']}-NB"
    else:
        return row['occtype']

In [6]:
def wseBuildingPts(tif_path, gdf_path):
    """Assigns WSE values to each building points"""
    rb, gt, src = getTifData(tif_path)
    wse_result = StructureData(gdf_path, gt, rb)
    return wse_result

In [7]:
def damage_pct(gdf, damage_df):
    """Reads building data (gdf) and depth damage function data (damage_ddf) and computes damage percentage values based on the 
    depth of water for each building points. This function is adjusted to work for Turkey and Brush Creek datasets.
    """
    gdf['dmg_pct'] = None
    
    # Loop through each row in the GeoDataFrame
    for idx, row in gdf.iterrows():
        occupancy_type = row['occ_new']  # Adjust column name as needed
        depth = row['depth']  # Adjust column name as needed

        # Check if depth is -9999, assign 0 damage and continue
        # This is done, because if a building point has a depth value of -9999, then there is no water in the waster surface raster. So, assigning zero damage
        if depth == -9999:
            gdf.at[idx, 'dmg_pct'] = 0
            continue
    
        # Get the matching row from the damage DataFrame
        damage_row = damage_df[damage_df['OccuNSI'] == occupancy_type]

        # Extract depth and damage values from the damage DataFrame
        depth_values = np.array([
        float(str(col)[1:]) * (-1 if str(col).startswith('m') else 1) if str(col) != '0' else 0
        for col in damage_row.columns[5:]
        ])
    
        damage_values = damage_row.iloc[0, 5:].values.astype(float)
    
        # Interpolate damage percentage for the given depth
        interpolated_damage = np.interp(depth, depth_values, damage_values)
    
        # Assign the interpolated damage percentage value to the GeoDataFrame
        gdf.at[idx, 'dmg_pct'] = interpolated_damage

    return gdf

In [8]:
def AAL_all_wse(tif_dir, bld_file, ddf_file, prob_csv):
    """Reads all the WSE tif files, probability weights and calculates 
    overall weighted damage values (AAL) in a single geodataframe"""

    final_gdf = gpd.read_file(bld_file)
    ddf = pd.read_excel(ddf_file)
    weights_df = pd.read_csv(weights_csv)
    weights_df.set_index('wse_rasters', inplace=True)

    # Get all TIFF files from the specified directory
    tif_files = glob.glob(os.path.join(tif_dir, "*.tif"))

    # Initialize a damage sum column
    final_gdf['dmg_sum'] = 0.0
    
    occtype_val = ["RES3A", "RES3B", "RES3C", "RES3D", "RES3E", "RES3F"]
    found_type_val = "B"

    for tif_file in tif_files:
        raster_name = os.path.splitext(os.path.basename(tif_file))[0]
        print(f"Processing: {raster_name}")

        # Get the probability weight from the CSV
        weight = weights_df.loc[raster_name, 'weight']

        # Calculate WSE and depth, adding two columns to the building GeoDataFrame
        wse_depth = wseBuildingPts(tif_file, bld_file)

        # Reclassify RES3 occupancy type based on basement or not
        #wse_depth['occ_new'] = wse_depth.apply(new_occtype, axis=1)
        wse_depth['occ_new'] = wse_depth.apply(new_occtype, args=(occtype_val, found_type_val), axis=1)

        wse_depth = damage_pct(wse_depth, ddf)                           # calculating damage percentage values

        wse_depth['weighted_dmg'] = wse_depth['dmg_pct'] * weight     # damage percentage * probability weight
        final_gdf['dmg_sum'] += wse_depth['weighted_dmg']                # sum of weighted damage percentage values from all WSE rasters into a new gdf
    final_gdf['occ_new'] = wse_depth['occ_new']
    final_gdf['dmg_val'] = (final_gdf['dmg_sum']/100) * final_gdf['val_struct']  # final building loss in US Dollars
        
    return final_gdf

In [9]:
# Final AAL calculation using all the functions defined above

root_dir = pl.Path(os.getcwd())

tif_directory = root_dir/'100yr_wse_rasters'
bld_file = root_dir/'shp_nsi/turkey_nsi_proj.shp'
ddf_file = root_dir/'ddf_final.xlsx'
weights_csv = root_dir/'event_weights_Turkey100.csv'

final_AAL = AAL_all_wse(tif_directory, bld_file, ddf_file, weights_csv)

Processing: 100_L_q1
Processing: 100_L_q2
Processing: 100_L_q3
Processing: 100_L_q4
Processing: 100_U_q1
Processing: 100_U_q2
Processing: 100_U_q3
Processing: 100_U_q4


In [10]:
# Save the final GeoDataFrame consisting of final AAL values

output_path_AAL = root_dir/'output/AAL_Turkey_100yr.shp'
final_AAL.to_file(output_path_AAL, driver='ESRI Shapefile')

## AEP raster calculation

In [11]:
def calculateAEP(tif_dir, csv_path):
    """Reads all WSE raster files and Probability weights csv. And computes AEP by
    assigning probability weights based on raster names"""
    
    weights_df = pd.read_csv(weights_csv)  # probability weights
    weights_df.set_index('wse_rasters', inplace=True) 

    # Getting all TIFF files
    tif_files = glob.glob(os.path.join(tif_dir, "*.tif"))
    
    aep_raster = None

    for tif_file in tif_files:
        raster_name = os.path.splitext(os.path.basename(tif_file))[0]
        print(f"Processing: {raster_name}")
        
        rb, gt, src = getTifData(tif_file)  # Get raster data
        wse = rb.ReadAsArray()
    
        # for first iteration
        if aep_raster is None:
            aep_raster = np.zeros_like(wse, dtype=np.float32)
    
        # Event probability weight from CSV file
        weight = weights_df.loc[raster_name, 'weight']
    
        # Applying the event probability weight where WSE value exists
        mask = wse > 0  # Masking to only get non-zero WSE values
        aep_raster[mask] += weight
    
    return aep_raster

In [12]:
def AEPasTIF(aep_raster, reference_tif, output_path):
    """Saves the computed AEP values as a TIFF file using the properties of a 
    reference TIF file (i.e., one of the output WSE tif file"""
    
    # Opening the reference TIFF to use the properties for the output raster
    rb, gt, src = getTifData(reference_tif)
    projection = src.GetProjection()
    geotransform = src.GetGeoTransform()
    
    rows, cols = aep_raster.shape      # dimensions of output raster

    # Creating a new TIFF file with same properties
    driver = gdal.GetDriverByName("GTiff")
    out_tif = driver.Create(str(output_path), cols, rows, 1, gdal.GDT_Float32)
    out_tif.SetGeoTransform(geotransform)
    out_tif.SetProjection(projection)

    # Writing AEP raster to the file
    out_tif.GetRasterBand(1).WriteArray(aep_raster)
    out_tif.GetRasterBand(1).SetNoDataValue(-9999)  # Setting NoData value
    out_tif.FlushCache()
    out_tif = None  

    print(f"AEP raster saved to: {output_path}")

In [13]:
# AEP raster calculation using above functions

tif_directory = root_dir/'100yr_wse_rasters'
weights_csv = root_dir/'event_weights_Turkey100.csv'

# calculating AEP values
aep_val = calculateAEP(tif_directory, weights_csv)

# Saving calculated AEP values as a tif file
ref_tif_files = glob.glob(os.path.join(tif_directory, "*.tif"))
reference_tif = ref_tif_files[0]
output_path_AEP = root_dir/'output/AEP_Turkey_100yr.tif'

AEPasTIF(aep_val, reference_tif, output_path_AEP)

Processing: 100_L_q1
Processing: 100_L_q2
Processing: 100_L_q3
Processing: 100_L_q4
Processing: 100_U_q1
Processing: 100_U_q2
Processing: 100_U_q3
Processing: 100_U_q4
AEP raster saved to: C:\Users\dneupane\Documents\Probabilistic\AAL_calc\output\AEP_Turkey_100yr.tif


### END